# 06 · Portfolio Transaction Simulation

Replays transactions through the FIFO lot engine, reconciles derived holdings
against a broker snapshot, and produces a per-security portfolio report.

**This notebook uses `src/portfolio_sim.py`.  
It does not import from `src/tax_risk_sim.py` or `src/inputs.py`.**

---

## Data sources

Set `MODE` in the next cell to choose the data source:

| Mode | Source | When to use |
|------|--------|-------------|
| `"pdf"` | Deutsche Bank PDF in `data/private/` | Real portfolio — parses transactions + holdings directly from the broker PDF |
| `"synthetic"` | Bundled CSV fixtures in `data/examples/` | Testing, CI, offline demos — no PDF or internet required |

**PDF mode** seeds the lot ledger from the holdings snapshot (covers positions
bought before the report period), then replays all transactions from the PDF on
top. Reconciliation confirms whether the lot engine's share counts match the
broker snapshot.

> **FX note**: PDF mode uses live ECB rates (internet required).
> Synthetic mode uses a fixed-rate stub (fully offline).

In [ ]:
import sys
from pathlib import Path

# ── Mode selector ─────────────────────────────────────────────────────────────
# "pdf"       → parse a Deutsche Bank PDF (real data, requires data/private/)
# "synthetic" → use bundled synthetic fixtures (no PDF needed, works offline)
MODE = "pdf"

WORK = Path(".").resolve()
if WORK.name == "notebooks":
    WORK = WORK.parent
SRC = WORK / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# ── PDF mode paths ────────────────────────────────────────────────────────────
PDF_PATH        = WORK / "data/private/report.pdf"
TICKER_MAP_PATH = WORK / "data/private/ticker_map.json"

# ── Synthetic mode paths ──────────────────────────────────────────────────────
TX_CSV      = WORK / "data/examples/db_transactions_synthetic.csv"
HLD_CSV     = WORK / "data/examples/db_holdings_synthetic.csv"
TX_PARQUET  = WORK / "data/examples/db_transactions_synthetic.parquet"
HLD_PARQUET = WORK / "data/examples/db_holdings_synthetic.parquet"

print(f"Mode      : {MODE}")
print(f"Repo root : {WORK}")

In [ ]:
import json
import subprocess

import pandas as pd

if MODE == "pdf":
    # ── Parse Deutsche Bank PDF ───────────────────────────────────────────────
    from pdf_parser import parse_db_pdf

    _tx_df, _hld_df = parse_db_pdf(PDF_PATH)
    transactions      = _tx_df
    holdings_snapshot = _hld_df

    with open(TICKER_MAP_PATH) as _f:
        _raw = json.load(_f)
    TICKER_MAP = {k: v for k, v in _raw.items() if not k.startswith("_")}

    REPORTING_DATE = str(_hld_df["date"].iloc[0])

    print(f"Transactions  : {len(transactions)} rows")
    print(f"  range       : {transactions['date'].min()} → {transactions['date'].max()}")
    print(f"Holdings      : {len(holdings_snapshot)} positions")
    print(f"Reporting date: {REPORTING_DATE}")
    print(f"Ticker map    : {len(TICKER_MAP)} ISINs")

else:
    # ── Normalize synthetic CSVs → Parquet ───────────────────────────────────
    NORMALIZE = WORK / "scripts/normalize_portfolio_inputs.py"
    import sys as _sys

    for csv_path, pq_path, schema_type in [
        (TX_CSV,  TX_PARQUET,  "transactions"),
        (HLD_CSV, HLD_PARQUET, "holdings"),
    ]:
        if not pq_path.exists():
            result = subprocess.run(
                [
                    _sys.executable, str(NORMALIZE),
                    "--input",          str(csv_path),
                    "--output-csv",     str(csv_path.with_suffix(".normalized.csv")),
                    "--output-parquet", str(pq_path),
                    "--type",           schema_type,
                ],
                capture_output=True, text=True,
            )
            if result.returncode != 0:
                print(f"ERROR: {result.stderr}")
            else:
                print(result.stdout.strip())
        else:
            print(f"Already exists: {pq_path.name}")

    transactions      = pd.read_parquet(TX_PARQUET)
    holdings_snapshot = pd.read_parquet(HLD_PARQUET)
    REPORTING_DATE    = "2023-12-31"
    TICKER_MAP        = {}

    print(f"\nTransactions : {len(transactions)} rows")
    print(f"Holdings     : {len(holdings_snapshot)} rows")

In [ ]:
print("Transactions:")
display(transactions)
print("\nHoldings snapshot:")
display(holdings_snapshot)

In [ ]:
from portfolio_sim import ECBProvider, FXProvider, make_fx_provider

if MODE == "pdf":
    # Live ECB rates — requires internet
    fx = make_fx_provider("ecb")
    print("FX provider: ECBProvider (live rates)")
else:
    # Fixed-rate stub — offline, reproducible for synthetic demos
    class FixedRateFXProvider(FXProvider):
        _RATES: dict[tuple[str, str], float] = {
            ("USD", "EUR"): 0.9200,
            ("EUR", "USD"): 1.0870,
            ("GBP", "EUR"): 1.1500,
            ("EUR", "GBP"): 0.8696,
        }

        def rate(self, from_currency: str, to_currency: str, date: str) -> float:
            if from_currency == to_currency:
                return 1.0
            key = (from_currency, to_currency)
            if key in self._RATES:
                return self._RATES[key]
            to_eur  = self.rate(from_currency, "EUR", date)
            eur_to  = self.rate("EUR", to_currency, date)
            return to_eur * eur_to

    fx = FixedRateFXProvider()
    print("FX provider: FixedRateFXProvider (offline demo)")
    print(f"  USD → EUR : {fx.rate('USD', 'EUR', '2023-12-31'):.4f}")

In [ ]:
from portfolio_sim import YahooPriceProvider, fetch_current_prices

CAPITAL_GAINS_TAX = 0.26375   # German Abgeltungsteuer 25% + Solidaritätszuschlag
DIVIDEND_TAX      = 0.26375

if MODE == "pdf":
    # Fetch live EUR prices for every holding via Yahoo Finance
    _price_provider = YahooPriceProvider(isin_to_ticker=TICKER_MAP, fx_provider=fx)
    _isins = holdings_snapshot["isin"].dropna().unique().tolist()
    CURRENT_PRICES_EUR = fetch_current_prices(_isins, _price_provider, REPORTING_DATE)
    _missing = [i for i in _isins if i not in CURRENT_PRICES_EUR]
    print(f"Prices fetched : {len(CURRENT_PRICES_EUR)}/{len(_isins)} ISINs")
    if _missing:
        print(f"  WARNING: no price for {_missing} — those positions will be excluded")
else:
    # Static prices for the synthetic demo (end-2023 approximations)
    _apple_eur = fx.rate("USD", "EUR", REPORTING_DATE) * 192.50
    CURRENT_PRICES_EUR = {
        "DE00SYNTH001": 15.60,
        "IE00B4L5Y983": 87.20,
        "US00SYNTH003": _apple_eur,
    }

print(f"\nReporting date     : {REPORTING_DATE}")
print(f"Capital gains tax  : {CAPITAL_GAINS_TAX:.5f}")
print(f"Dividend tax       : {DIVIDEND_TAX:.5f}")
print("\nCurrent prices (EUR):")
for isin, price in CURRENT_PRICES_EUR.items():
    print(f"  {isin}: {price:.4f}")

In [6]:
# ── Step 5: check for unsupported corporate actions ───────────────────────────

from portfolio_sim import check_unsupported_actions

blocked_isins = check_unsupported_actions(transactions)

if blocked_isins:
    print(f'WARNING: {len(blocked_isins)} ISIN(s) blocked by unsupported corporate actions:')
    for isin in blocked_isins:
        print(f'  - {isin}')
    print('Partial results will exclude these securities.')
else:
    print('No unsupported corporate actions — full simulation available.')

No unsupported corporate actions — full simulation available.


In [7]:
# ── Step 6: run portfolio simulation ─────────────────────────────────────────
#
# simulate_portfolio_partial is the safe default: excludes blocked ISINs instead
# of raising UnsupportedCorporateAction. With no blocked ISINs it is equivalent
# to simulate_portfolio.

from portfolio_sim import simulate_portfolio_partial

result, excluded = simulate_portfolio_partial(
    transactions=transactions,
    current_prices_eur=CURRENT_PRICES_EUR,
    capital_gains_tax_rate=CAPITAL_GAINS_TAX,
    dividend_tax_rate=DIVIDEND_TAX,
    fx_provider=fx,
    lot_method='fifo',
    reporting_date=REPORTING_DATE,
)

if excluded:
    print(f'PARTIAL RESULTS — excluded ISINs (unsupported actions): {excluded}')
    print('Totals below do NOT represent the full portfolio.\n')

display(result)

,isin,reporting_date,market_value_eur,unrealised_gain_eur,realised_gain_ytd_eur,tax_paid_ytd_eur
0,DE00SYNTH001,2023-12-31,4290.0,760.0,802.00,240.65
1,IE00B4L5Y983,2023-12-31,0.0,0.0,0.00,0.00
2,IE00SYNTH002,2023-12-31,0.0,-2562.0,0.00,0.00
3,US00SYNTH003,2023-12-31,1771.0,161.0,8.83,2.33


In [8]:
# ── Step 7: portfolio totals ──────────────────────────────────────────────────

totals = result[[
    'market_value_eur',
    'unrealised_gain_eur',
    'realised_gain_ytd_eur',
    'tax_paid_ytd_eur',
]].sum()

print(f'Portfolio report — {REPORTING_DATE}')
print('=' * 45)
print(f'  Market value (EUR)      : {totals["market_value_eur"]:>12,.2f}')
print(f'  Unrealised gain (EUR)   : {totals["unrealised_gain_eur"]:>12,.2f}')
print(f'  Realised gain YTD (EUR) : {totals["realised_gain_ytd_eur"]:>12,.2f}')
print(f'  Tax paid YTD (EUR)      : {totals["tax_paid_ytd_eur"]:>12,.2f}')

if excluded:
    print()
    print(f'  PARTIAL — {len(excluded)} ISIN(s) excluded: {excluded}')

Portfolio report — 2023-12-31
  Market value (EUR)      :     6,061.00
  Unrealised gain (EUR)   :    -1,641.00
  Realised gain YTD (EUR) :       810.83
  Tax paid YTD (EUR)      :       242.98


In [9]:
# ── Step 8: holdings reconciliation ──────────────────────────────────────────
#
# Compare transaction-derived share counts against the broker snapshot.
# Status legend:
#   match        — derived vs broker difference <= 0.001 shares
#   mismatch     — difference > tolerance (investigate before trusting output)
#   derived_only — in transactions but not in broker snapshot
#   broker_only  — in broker snapshot but not in transactions

from portfolio_sim import (
    apply_buy,
    apply_sell_fifo,
    apply_split,
    derive_holdings_from_lots,
    reconcile_holdings,
)

# Replay transactions to rebuild the lot ledger for reconciliation
# (simulate_portfolio returns the output DataFrame, not the internal lots)
lots: list[dict] = []
for _, row in transactions.sort_values('date').iterrows():
    isin     = row['isin']
    tx_type  = row['transaction_type']
    currency = str(row['currency'])
    date     = str(row['date'])

    if tx_type == 'buy':
        price_eur = fx.convert(float(row['price']), currency, 'EUR', date)
        lots = apply_buy(lots, isin, date, price_eur, float(row['quantity']))
    elif tx_type == 'sell':
        price_eur = fx.convert(float(row['price']), currency, 'EUR', date)
        lots, _, _ = apply_sell_fifo(lots, isin, float(row['quantity']), price_eur, CAPITAL_GAINS_TAX)
    elif tx_type == 'split':
        lots = apply_split(lots, isin, float(row['quantity']))

lot_holdings    = derive_holdings_from_lots(lots)
broker_holdings = holdings_snapshot[['isin', 'quantity']].copy()

reconciliation = reconcile_holdings(lot_holdings, broker_holdings)
print('Holdings reconciliation:')
display(reconciliation)

mismatches = reconciliation[reconciliation['status'] == 'mismatch']
if mismatches.empty:
    print('All holdings match the broker snapshot (within tolerance).')
else:
    print(f'WARNING: {len(mismatches)} mismatch(es) — review before trusting simulation output.')
    display(mismatches)

Holdings reconciliation:


,isin,derived_quantity,broker_quantity,difference,status
0,DE00SYNTH001,275.0,275,0.0,match
1,IE00SYNTH002,30.0,30,0.0,match
2,US00SYNTH003,10.0,10,0.0,match


All holdings match the broker snapshot (within tolerance).


In [10]:
# ── Step 9: lot ledger with derived unrealised gain ───────────────────────────

from portfolio_sim import lots_to_dataframe

lot_df = lots_to_dataframe(lots)
lot_df['unrealised_gain_eur'] = lot_df.apply(
    lambda r: round(
        (CURRENT_PRICES_EUR.get(r['isin'], 0.0) - r['lot_price_eur']) * r['remaining_shares'], 2
    ),
    axis=1,
)

print('Open lot ledger with derived unrealised gain (computed at reporting date, not stored):')
display(lot_df)

Open lot ledger with derived unrealised gain (computed at reporting date, not stored):


,isin,lot_date,lot_price_eur,remaining_shares,unrealised_gain_eur
0,DE00SYNTH001,2022-03-10,11.2,50.0,220.0
1,DE00SYNTH001,2022-08-22,12.8,100.0,280.0
2,DE00SYNTH001,2023-01-15,12.5,50.0,155.0
3,DE00SYNTH001,2023-06-01,14.2,75.0,105.0
4,IE00SYNTH002,2023-11-10,85.4,30.0,-2562.0
5,US00SYNTH003,2023-11-10,161.0,10.0,161.0


## Known limitations (v1)

| Limitation | Note |
|------------|------|
| Flat tax rates | `CAPITAL_GAINS_TAX` and `DIVIDEND_TAX` applied uniformly. German Sparer-Pauschbetrag (€1,000/year exemption) is not modelled. |
| Solidarity surcharge | Not modelled separately — fold it into the flat rate if needed (26.375% = 25% × 1.055). |
| Reverse-split fractional shares | Retained at full precision; no cash distribution event generated. |
| Unsupported corporate actions | `merger`, `spin_off`, `option` block the affected ISIN from trusted totals. |
| Jurisdiction-aware tax | `jurisdiction` field captured but not yet used to vary tax logic by country. |
| Transaction-date FX | FX conversion uses the transaction date; intraday spot rates are not modelled. |

See `docs/portfolio-transaction-mvp/requirements-and-decisions.md` for full decisions and limitations.